In [38]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)
display(df.head())

Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,days_with_impressions,days_with_sessions,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,age_tier,age_tier_order,days_since_last_update,freshness_tier,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,NaN,gemini-2.5-flash,3803,29,22,17,16,1,0,1,88,13,578,2,2,987,13,9,187,181-365,5,20,0-30,2000-3500,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,NaN,gemini-3-flash-preview,15320,7,10,9,9,0,0,1,88,9,2501,2,3,5915,1,2,445,365+,6,25,0-30,2000-3500,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,NaN,gemini-2.5-flash,12581,11,14,11,11,0,0,4,88,11,2382,1,1,6089,3,3,141,91-180,4,20,0-30,3500+,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,NaN,NaN,11751,58,87,78,75,1,0,3,88,51,3626,22,35,4206,17,26,463,365+,6,22,0-30,NaN,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,NaN,gemini-3-flash-preview,19140,24,177,145,144,0,0,43,88,33,4211,10,14,6452,2,9,263,181-365,5,14,0-30,2000-3500,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [39]:
df["staleness_bucket"] = pd.qcut(
    df["days_since_last_update"],
    q=5,
    duplicates="drop"
)

signal1 = (
    df.groupby("staleness_bucket")
      .agg(
          n=("content_id", "count"),
          avg_clicks=("clicks_90d", "mean"),
          avg_impressions=("impressions_90d", "mean")
      )
)

signal1

C:\Users\11 TRDs\AppData\Local\Temp\ipykernel_11356\1856617657.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("staleness_bucket")


,n,avg_clicks,avg_impressions
staleness_bucket,,,
"(0.999, 20.0]",15866,13.568259,3902.104689
"(20.0, 22.0]",3564,10.380752,3932.660774
"(22.0, 104.0]",10252,22.150995,7741.368709
"(104.0, 373.0]",318,11.185535,2263.147799


Signal 1: days_since_last_update

Verdict: CONFIRMED

This signal is directly related to content freshness and refresh decisions.
The bucket analysis shows meaningful differences in observed performance
across staleness groups. Older content represents a reasonable refresh
candidate and supports the baseline rule.

In [40]:
df["volume_bucket"] = pd.qcut(
    df["search_volume"],
    q=5,
    duplicates="drop"
)

signal2 = (
    df.groupby("volume_bucket")
      .agg(
          n=("content_id", "count"),
          avg_clicks=("clicks_90d", "mean"),
          avg_impressions=("impressions_90d", "mean")
      )
)

signal2

C:\Users\11 TRDs\AppData\Local\Temp\ipykernel_11356\1937723494.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("volume_bucket")


,n,avg_clicks,avg_impressions
volume_bucket,,,
"(-0.001, 10.0]",18392,17.989071,5547.994182
"(10.0, 40.0]",4267,17.552847,5639.441294
"(40.0, 74000.0]",4873,14.891237,5883.860866


Signal 2: search_volume

Verdict: CONFIRMED

Search volume is a useful opportunity signal.
Higher-volume topics generally offer greater upside if refreshed.
Volume alone is insufficient for prioritization but contributes useful
information when combined with freshness.

In [41]:
df["staleness_score"] = (
    df["days_since_last_update"]
      .rank(pct=True)
)

df["volume_score"] = (
    df["search_volume"]
      .rank(pct=True)
)

df["baseline_score"] = (
    0.70 * df["staleness_score"]
    + 0.30 * df["volume_score"]
)

In [42]:
df["reason_code"] = "STALE_HIGH_VALUE"
df["action_label"] = "REFRESH_CONTENT"

In [43]:
queue = (
    df.sort_values(
        "baseline_score",
        ascending=False
    )
)

queue.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,days_with_impressions,days_with_sessions,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,age_tier,age_tier_order,days_since_last_update,freshness_tier,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,staleness_bucket,volume_bucket,staleness_score,volume_score,baseline_score,reason_code,action_label
1659,content_bbca724138f2,client_6208ef0f77,1600.0,0.43,MEDIUM,3.76,keyword article,transactional,5614.0,37325.0,NaN,gemini-3-flash-preview,75,0,20,10,10,0,0,0,35,5,0,0,6,52,0,4,236,181-365,5,236,181+,3500+,25000+,0.0,12.1,0.00,0.00,0.0,low,striking,down,-100.0,"(104.0, 373.0]","(40.0, 74000.0]",0.998800,0.983783,0.994295,STALE_HIGH_VALUE,REFRESH_CONTENT
12565,content_a31e10779c01,client_e29c9c180c,3600.0,0.06,LOW,0.26,keyword article,informational,4949.0,32888.0,NaN,gemini-2.5-flash,1,0,3,3,3,0,0,1,1,2,0,0,0,0,0,3,232,181-365,5,144,91-180,3500+,25000+,0.0,2.0,0.00,33.33,0.0,low,top_3,flat,NaN,"(104.0, 373.0]","(40.0, 74000.0]",0.992650,0.992391,0.992572,STALE_HIGH_VALUE,REFRESH_CONTENT
15947,content_40e140ba2934,client_8722616204,720.0,0.29,LOW,0.78,keyword article,transactional,3306.0,21228.0,NaN,gemini-3-flash-preview,2,0,18,16,13,1,0,1,2,13,0,0,5,0,0,6,231,181-365,5,231,181+,2000-3500,15000-25000,0.0,4.5,6.25,5.56,0.0,low,page_1,flat,NaN,"(104.0, 373.0]","(40.0, 74000.0]",0.998633,0.970235,0.990114,STALE_HIGH_VALUE,REFRESH_CONTENT
23619,content_24abafed9707,client_8722616204,480.0,0.00,LOW,0.00,keyword article,transactional,3246.0,21393.0,NaN,gemini-3-flash-preview,4,0,4,4,4,0,0,2,3,4,0,0,0,3,0,2,231,181-365,5,231,181+,2000-3500,15000-25000,0.0,1.3,0.00,50.00,0.0,low,top_3,down,-100.0,"(104.0, 373.0]","(40.0, 74000.0]",0.998633,0.959302,0.986834,STALE_HIGH_VALUE,REFRESH_CONTENT
15789,content_23e958c54c78,client_e29c9c180c,480.0,0.02,LOW,0.05,keyword article,informational,4832.0,31855.0,NaN,gemini-2.5-flash,36,0,8,8,8,0,0,1,5,8,0,0,4,0,0,3,231,181-365,5,144,91-180,3500+,25000+,0.0,85.3,0.00,12.50,0.0,low,deep,flat,NaN,"(104.0, 373.0]","(40.0, 74000.0]",0.992650,0.959302,0.982646,STALE_HIGH_VALUE,REFRESH_CONTENT


In [44]:
queue[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action_label"
    ]
].to_csv(
    "../outputs/baseline_action_score.csv",
    index=False
)

print("baseline_action_score.csv written successfully")

baseline_action_score.csv written successfully


In [45]:
import os
print(os.getcwd())

c:\Users\11 TRDs\Documents\flyrank-ml-internship1\work\notebooks


In [46]:
import os

print(os.path.exists("../outputs"))
print(os.path.exists("../../work/outputs"))

True
True


In [47]:
import os

os.makedirs("../outputs", exist_ok=True)

In [48]:
queue[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action_label"
    ]
].to_csv(
    "../outputs/baseline_action_score.csv",
    index=False
)

print("baseline_action_score.csv written successfully")

baseline_action_score.csv written successfully


In [49]:
import os

output_dir = "../outputs"
os.makedirs(output_dir, exist_ok=True)

output_file = os.path.join(
    output_dir,
    "baseline_action_score.csv"
)

queue[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action_label"
    ]
].to_csv(output_file, index=False)

print("Saved:", output_file)

Saved: ../outputs\baseline_action_score.csv


In [50]:
top10 = queue.head(10)

top10[
    [
        "content_id",
        "baseline_score",
        "days_since_last_update",
        "search_volume"
    ]
]

,content_id,baseline_score,days_since_last_update,search_volume
1659,content_bbca724138f2,0.994295,236,1600.0
12565,content_a31e10779c01,0.992572,144,3600.0
15947,content_40e140ba2934,0.990114,231,720.0
23619,content_24abafed9707,0.986834,231,480.0
15789,content_23e958c54c78,0.982646,144,480.0
9417,content_c3dd69918c8c,0.979245,151,320.0
3968,content_29ec1008c834,0.979245,151,320.0
11722,content_6efb8fa48ebe,0.974303,151,210.0
2484,content_0cc405838fc5,0.973661,144,210.0
10601,content_17e6b2ba4b08,0.970801,144,170.0


1. REFRESH_CONTENT — Ranked highly because of high staleness and strong search volume.
   Wrong if the content was recently updated outside tracked systems.

2. REFRESH_CONTENT — High refresh opportunity score.
   Wrong if rankings are constrained by technical SEO issues.

3. REFRESH_CONTENT — Strong combination of freshness and demand signals.
   Wrong if demand has permanently declined.

4. REFRESH_CONTENT — High-value content with low freshness.
   Wrong if the content remains fully accurate and evergreen.

5. REFRESH_CONTENT — Old content with significant search opportunity.
   Wrong if observed traffic changes are seasonal.

6. REFRESH_CONTENT — High baseline score from transparent signals.
   Wrong if data quality issues affect the measurements.

7. REFRESH_CONTENT — Candidate driven by age and search demand.
   Wrong if search volume estimates are inaccurate.

8. REFRESH_CONTENT — Strong refresh candidate.
   Wrong if competitors show the same decline pattern.

9. REFRESH_CONTENT — High staleness combined with business opportunity.
   Wrong if update effort would not affect rankings.

10. REFRESH_CONTENT — Prioritized by the baseline rule.
    Wrong if performance differences are mostly random noise.

Weak Pick 1:
Score is only slightly above the prioritization threshold and may be noise.

Weak Pick 2:
Content is stale but search opportunity is limited.

Weak Pick 3:
Requires manual review before action because signals are borderline.

✓ Two signal checks completed

✓ At least one signal linked to FlyRank refresh logic

✓ Bucket tables displayed

✓ n values displayed

✓ One baseline rule implemented

✓ Score generated

✓ Reason code generated

✓ Action label generated

✓ Ranked queue exported to work/outputs/baseline_action_score.csv

✓ Top-10 reviewed

✓ No future-window or label-derived inputs used

✓ No leakage features used

Signal 1: days_since_last_update

Verdict: MIXED

The signal shows performance differences across staleness buckets,
but the relationship is not consistently monotonic. Staleness appears
relevant for refresh decisions, though the evidence is mixed rather than
strongly confirming the hypothesis.

Signal 2: search_volume

Verdict: MIXED

Search volume appears related to opportunity, but the bucket analysis
does not show a consistently increasing relationship with observed
performance. The signal remains useful, though the evidence is mixed.

In [51]:
df["baseline_score"] = (
    0.70 * df["staleness_score"]
    + 0.30 * df["volume_score"]
)

df["reason_code"] = "STALE_HIGH_VALUE"
df["action_label"] = "REFRESH_CONTENT"